# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

The action queue is a decision-support output, not an automatic publishing decision.

The queue ranks observations using the validated model score from ML-09 and attaches simple reason codes that a human reviewer can understand. The main reasons are based on observable current performance signals: declining clicks, low CTR, and weak search position.

The highest-ranked observations should be reviewed first because they combine a higher model score with one or more observable warning signals.

The reason codes are:

- `TREND_DOWN` — current GSC clicks are lower than the previous observed day.
- `LOW_CTR` — CTR is at or below the training-derived lower-quartile threshold.
- `WEAK_POSITION` — average search position is at or above the training-derived upper-quartile threshold.
- `MODEL_PRIORITY` — the Random Forest assigns a high relative ranking score.

These codes explain why an observation appears in the queue; they do not prove that a content refresh will improve performance.

The recommended action is therefore **"REVIEW"**, rather than automatically "REFRESH", "PRUNE", or "REWRITE".

In [2]:
from getpass import getpass

import duckdb
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

REL = "hf://datasets/FlyRank/internship-warehouse"

PERF_JAN = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2025-01/*.parquet'"
    f")"
)

data = con.sql(
    f"SELECT * FROM {PERF_JAN}"
).df()

data["report_date"] = pd.to_datetime(
    data["report_date"]
)

print("Shape:", data.shape)
print("Date range:",
      data["report_date"].min().date(),
      "to",
      data["report_date"].max().date())

print("\nRows by date:")
print(
    data["report_date"]
    .value_counts()
    .sort_index()
)

Shape: (1297, 31)
Date range: 2025-01-27 to 2025-01-31

Rows by date:
report_date
2025-01-27    303
2025-01-28    317
2025-01-29    262
2025-01-30    194
2025-01-31    221
Name: count, dtype: int64


In [5]:
group_cols = [
    "client_hash_id",
    "content_hash_id"
]

duplicate_grain = (
    data.groupby(
        group_cols + ["report_date"]
    )
    .size()
)

print(
    "Duplicate client-content-date rows:",
    (duplicate_grain > 1).sum()
)

assert (duplicate_grain > 1).sum() == 0

Duplicate client-content-date rows: 0


In [6]:
data = data.sort_values(
    group_cols + ["report_date"]
).copy()

data["next_date"] = (
    data.groupby(group_cols)["report_date"]
    .shift(-1)
)

data["days_to_next"] = (
    data["next_date"] -
    data["report_date"]
).dt.days

data["next_day_clicks"] = (
    data.groupby(group_cols)["gsc_clicks"]
    .shift(-1)
)

data.loc[
    data["days_to_next"] != 1,
    "next_day_clicks"
] = np.nan

print("Days to next observation:")
print(
    data["days_to_next"]
    .value_counts(dropna=False)
    .sort_index()
)

model_data = data.dropna(
    subset=["next_day_clicks"]
).copy()

print(
    "\nValid next-day target rows:",
    len(model_data)
)

Days to next observation:
days_to_next
1.0    780
2.0     34
3.0      5
4.0      2
NaN    476
Name: count, dtype: int64

Valid next-day target rows: 780


In [7]:
model_data["ctr"] = np.where(
    model_data["gsc_impressions"] > 0,
    model_data["gsc_clicks"]
    / model_data["gsc_impressions"],
    np.nan
)

model_data["total_sessions"] = (
    model_data["sessions_organic"].fillna(0)
    + model_data["sessions_direct"].fillna(0)
    + model_data["sessions_referral"].fillna(0)
    + model_data["sessions_social"].fillna(0)
    + model_data["sessions_paid"].fillna(0)
    + model_data["sessions_ai"].fillna(0)
)

model_data["organic_share"] = np.where(
    model_data["total_sessions"] > 0,
    model_data["sessions_organic"]
    / model_data["total_sessions"],
    np.nan
)

features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
]

X = model_data[features].copy()
y = model_data["next_day_clicks"].copy()

# Random Forest does not accept NaN.
X = X.fillna(0)

print("Number of features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 14
X shape: (780, 14)
y shape: (780,)


In [8]:
def make_model():
    return RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

print(make_model())

RandomForestRegressor(max_depth=6, min_samples_leaf=5, n_estimators=300,
                      n_jobs=-1, random_state=42)


In [9]:
random_train_idx, random_test_idx = train_test_split(
    np.arange(len(model_data)),
    test_size=0.20,
    random_state=42
)

X_random_train = X.iloc[random_train_idx]
X_random_test = X.iloc[random_test_idx]

y_random_train = y.iloc[random_train_idx]
y_random_test = y.iloc[random_test_idx]

random_model = make_model()

random_model.fit(
    X_random_train,
    y_random_train
)

random_pred = random_model.predict(
    X_random_test
)

random_ndcg = ndcg_score(
    [y_random_test.to_numpy()],
    [random_pred],
    k=min(10, len(y_random_test))
)

print("Random-split rows:")
print("Train:", len(X_random_train))
print("Test :", len(X_random_test))

print(
    "\nRandom-split NDCG@10:",
    round(random_ndcg, 6)
)

Random-split rows:
Train: 624
Test : 156

Random-split NDCG@10: 0.207261


In [10]:
train_end = pd.Timestamp(
    "2025-01-29"
)

time_train_mask = (
    model_data["report_date"]
    <= train_end
)

time_test_mask = (
    model_data["report_date"]
    > train_end
)

X_time_train = X.loc[time_train_mask]
X_time_test = X.loc[time_test_mask]

y_time_train = y.loc[time_train_mask]
y_time_test = y.loc[time_test_mask]

time_model = make_model()

time_model.fit(
    X_time_train,
    y_time_train
)

time_pred = time_model.predict(
    X_time_test
)

time_ndcg = ndcg_score(
    [y_time_test.to_numpy()],
    [time_pred],
    k=min(10, len(y_time_test))
)

print("Time-aware rows:")
print("Train:", len(X_time_train))
print("Test :", len(X_time_test))

print(
    "\nTime-aware NDCG@10:",
    round(time_ndcg, 6)
)

print(
    "\nTraining period:",
    model_data.loc[
        time_train_mask,
        "report_date"
    ].min().date(),
    "to",
    model_data.loc[
        time_train_mask,
        "report_date"
    ].max().date()
)

print(
    "Evaluation period:",
    model_data.loc[
        time_test_mask,
        "report_date"
    ].min().date(),
    "to",
    model_data.loc[
        time_test_mask,
        "report_date"
    ].max().date()
)

Time-aware rows:
Train: 600
Test : 180

Time-aware NDCG@10: 0.689987

Training period: 2025-01-27 to 2025-01-29
Evaluation period: 2025-01-30 to 2025-01-30


In [11]:
validation_comparison = pd.DataFrame({
    "split": [
        "Random split",
        "Time-aware split"
    ],
    "NDCG@10": [
        random_ndcg,
        time_ndcg
    ]
})

display(validation_comparison)

,split,NDCG@10
0,Random split,0.207261
1,Time-aware split,0.689987


In [13]:
train_data = model_data.loc[
    time_train_mask
].copy()

eval_data = model_data.loc[
    time_test_mask
].copy()

ctr_threshold = train_data[
    "ctr"
].quantile(0.25)

position_threshold = train_data[
    "gsc_avg_position"
].quantile(0.75)

print("Training-derived thresholds:")
print(
    "CTR:",
    ctr_threshold
)

print(
    "Position:",
    position_threshold
)

Training-derived thresholds:
CTR: 0.0
Position: 48.962500000000006


In [15]:
history = data.sort_values(
    group_cols + ["report_date"]
).copy()

history["previous_day_clicks"] = (
    history.groupby(group_cols)["gsc_clicks"]
    .shift(1)
)

previous_clicks = history[
    group_cols
    + ["report_date", "previous_day_clicks"]
].copy()

eval_data = eval_data.merge(
    previous_clicks,
    on=group_cols + ["report_date"],
    how="left",
    sort=False
)

In [16]:
eval_data["baseline_score"] = 0

# Declining current-day clicks versus previous day.
trend_down = (
    eval_data["previous_day_clicks"].notna()
    & (
        eval_data["gsc_clicks"]
        < eval_data["previous_day_clicks"]
    )
)

eval_data.loc[
    trend_down,
    "baseline_score"
] += 3

# Low CTR.
eval_data.loc[
    eval_data["ctr"] <= ctr_threshold,
    "baseline_score"
] += 2

# Weak observed position.
eval_data.loc[
    eval_data["gsc_avg_position"]
    >= position_threshold,
    "baseline_score"
] += 2

print(
    eval_data["baseline_score"]
    .value_counts()
    .sort_index()
)

baseline_score
0     16
2    105
4     47
5     12
Name: count, dtype: int64


In [17]:
eval_data["model_score"] = time_pred

assert len(eval_data) == len(time_pred)
assert eval_data["model_score"].notna().all()

In [18]:
y_true = eval_data[
    "next_day_clicks"
].to_numpy()

baseline_scores = eval_data[
    "baseline_score"
].to_numpy()

model_scores = eval_data[
    "model_score"
].to_numpy()

k = min(
    10,
    len(eval_data)
)

baseline_ndcg = ndcg_score(
    [y_true],
    [baseline_scores],
    k=k
)

model_ndcg = ndcg_score(
    [y_true],
    [model_scores],
    k=k
)

comparison = pd.DataFrame({
    "method": [
        "Rule baseline",
        "Random Forest"
    ],
    "NDCG@10": [
        baseline_ndcg,
        model_ndcg
    ]
})

display(comparison)

,method,NDCG@10
0,Rule baseline,0.066051
1,Random Forest,0.689987


In [19]:
# ML-10 — Section 1: Build ranked action queue

queue = eval_data.copy()

queue["reason_codes"] = ""

def add_reason(row):
    reasons = []

    if (
        pd.notna(row["previous_day_clicks"])
        and row["gsc_clicks"] < row["previous_day_clicks"]
    ):
        reasons.append("TREND_DOWN")

    if row["ctr"] <= ctr_threshold:
        reasons.append("LOW_CTR")

    if row["gsc_avg_position"] >= position_threshold:
        reasons.append("WEAK_POSITION")

    return "|".join(reasons) if reasons else "MODEL_PRIORITY"

queue["reason_codes"] = queue.apply(add_reason, axis=1)

queue["recommended_action"] = "REVIEW"

queue = queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)

action_queue = queue[
    [
        "priority_rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "model_score",
        "gsc_clicks",
        "next_day_clicks",
        "ctr",
        "gsc_avg_position",
        "reason_codes",
        "recommended_action",
    ]
].copy()

display(action_queue.head(20))

,priority_rank,client_hash_id,content_hash_id,report_date,model_score,gsc_clicks,next_day_clicks,ctr,gsc_avg_position,reason_codes,recommended_action
0,1,client_9958f0a7ae1df715,content_f94fe855380e150f,2025-01-30,2.135390,6,5.0,0.024291,1.655870,MODEL_PRIORITY,REVIEW
1,2,client_9958f0a7ae1df715,content_37b3bafd5f88fdd1,2025-01-30,2.031996,4,4.0,0.019802,5.034653,MODEL_PRIORITY,REVIEW
2,3,client_9958f0a7ae1df715,content_de7b08874af74c00,2025-01-30,1.488334,0,0.0,0.000000,7.160920,LOW_CTR,REVIEW
3,4,client_9958f0a7ae1df715,content_d02be57d816cf3d7,2025-01-30,1.107518,1,0.0,0.012821,3.564103,MODEL_PRIORITY,REVIEW
4,5,client_9958f0a7ae1df715,content_84f5a9ecfefa108e,2025-01-30,0.757609,1,1.0,0.014085,3.239437,MODEL_PRIORITY,REVIEW
5,6,client_9958f0a7ae1df715,content_f3a75d8cf58dd50b,2025-01-30,0.629907,1,0.0,0.015625,2.343750,MODEL_PRIORITY,REVIEW
6,7,client_9958f0a7ae1df715,content_4dbfdffce9a25bc0,2025-01-30,0.499433,0,0.0,0.000000,11.600000,LOW_CTR,REVIEW
7,8,client_9958f0a7ae1df715,content_dc40416a3cf75e56,2025-01-30,0.440211,1,0.0,0.033333,5.833333,MODEL_PRIORITY,REVIEW
8,9,client_9958f0a7ae1df715,content_aa8aa2fb530f0af0,2025-01-30,0.395397,0,0.0,0.000000,8.928571,LOW_CTR,REVIEW
9,10,client_9958f0a7ae1df715,content_6c7a4022b0992856,2025-01-30,0.388381,0,0.0,0.000000,22.033333,LOW_CTR,REVIEW


In [20]:
print("Queue rows:", len(action_queue))
print("\nReason-code distribution:")

print(
    action_queue["reason_codes"]
    .value_counts()
)

Queue rows: 180

Reason-code distribution:
reason_codes
LOW_CTR                  105
LOW_CTR|WEAK_POSITION     47
MODEL_PRIORITY            16
TREND_DOWN|LOW_CTR        12
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

The playbook is intended for SEO/content teams that need to decide which observations deserve human review first.

A reviewer can use the ranked queue to inspect pages showing weak or declining search-performance signals. The model provides a prioritization signal; the reason codes provide a simple explanation for the ranking.

The intended workflow is:

1. Rank observations.
2. Review the highest-priority observations.
3. Inspect the underlying page and search context.
4. Decide whether the appropriate action is refresh, expand, protect, monitor, or no action.
5. Record the human decision and, where possible, the subsequent outcome.

### Limits

The current model was evaluated on a short January window and uses next-day GSC clicks as a proxy outcome. Therefore, the queue should not be interpreted as a validated "pages that need refreshing" classifier.

The model has not established that a recommended refresh causes improved traffic or rankings.

The evaluation target is also highly sparse: 95% of the time-aware evaluation observations had zero next-day clicks.

The queue should therefore be treated as directional decision-support evidence. It should not be used as an autonomous content-management system.

In [22]:
zero_click_rate = (
    y_time_test.eq(0).mean()
)

print(
    "Evaluation observations:",
    len(y_time_test)
)

print(
    "Zero next-day-click observations:",
    int(y_time_test.eq(0).sum())
)

print(
    "Zero next-day-click proportion:",
    round(zero_click_rate, 4)
)

print(
    "\nNext-day click summary:"
)

print(
    y_time_test.describe()
)

Evaluation observations: 180
Zero next-day-click observations: 171
Zero next-day-click proportion: 0.95

Next-day click summary:
count    180.000000
mean       0.105556
std        0.554285
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        5.000000
Name: next_day_clicks, dtype: float64


In [23]:
# Section 2 — Basic queue population checks

print("Intended-use population:")
print("Queue rows:", len(action_queue))

print(
    "Unique client-content observations:",
    action_queue[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)

print(
    "\nRecommended action distribution:"
)

print(
    action_queue["recommended_action"]
    .value_counts()
)

print(
    "\nEvaluation zero-target proportion:",
    round(zero_click_rate, 4)
)

assert len(action_queue) > 0
assert action_queue["recommended_action"].eq("REVIEW").all()

Intended-use population:
Queue rows: 180
Unique client-content observations: 180

Recommended action distribution:
recommended_action
REVIEW    180
Name: count, dtype: int64

Evaluation zero-target proportion: 0.95


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

### Human review checklist

Before taking action on a ranked observation, a human reviewer should check:

1. **Search intent** — Does the page still match the intent behind the queries it serves?
2. **Content quality** — Is the information accurate, complete, useful, and current?
3. **Search position** — Is the observed position meaningful, or is the page receiving too little search exposure to interpret the signal reliably?
4. **CTR context** — Could the CTR be explained by position, SERP features, branding, or query intent rather than content quality?
5. **Recent changes** — Has the page, site, URL, or search environment recently changed?
6. **Business importance** — Is the page strategically important enough to justify editorial effort?
7. **Technical issues** — Are indexing, canonicalization, redirects, or other technical factors affecting performance?
8. **Evidence across time** — Is the observed weakness persistent rather than a one-day fluctuation?

### No-go list

The system should never automatically:

- publish or modify content;
- delete or redirect a page;
- declare a page "bad" or "low quality";
- claim that a refresh will increase traffic;
- make a causal claim from the model score;
- override an editorial or technical review;
- use a model score as the sole reason for a high-impact SEO decision.

The model identifies observations that may deserve attention. The final action remains a human decision.

In [24]:
# Section 3 — Verify that the queue contains review signals,
# not autonomous high-impact actions.

no_go_actions = [
    "AUTO_REFRESH",
    "AUTO_DELETE",
    "AUTO_REDIRECT",
    "AUTO_PUBLISH"
]

assert not action_queue[
    "recommended_action"
].isin(no_go_actions).any()

print("No-go automation check: PASSED")

print(
    "\nAll queue recommendations are:",
    action_queue["recommended_action"].unique()
)

print(
    "\nHuman review is required before action."
)

No-go automation check: PASSED

All queue recommendations are: <ArrowStringArray>
['REVIEW']
Length: 1, dtype: str

Human review is required before action.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



The recommendations should be considered stale when the data distribution or model ranking behavior changes materially.

### Monitoring signals

The following should be monitored on future evaluation windows:

- NDCG@10 of the model versus the rule baseline.
- Distribution of next-day clicks and the proportion of zero targets.
- Distribution of CTR and average position.
- Feature distributions, especially impressions and average position.
- Percentage of observations receiving each reason code.
- Model feature importance.
- Number of observations entering the review queue.

### Retrain / revalidation triggers

The model should be revalidated or retrained when:

1. NDCG@10 falls materially below the current validated result.
2. The model no longer outperforms the rule baseline.
3. The target distribution changes substantially.
4. CTR, impressions, or position distributions shift materially.
5. The client/content population changes substantially.
6. The data pipeline or measurement definitions change.
7. A longer evaluation period becomes available.

A single poor day should not automatically trigger retraining. The change should be confirmed over a sufficiently large evaluation window.

Future validation should use additional time periods and, where possible, a more meaningful sustained-decay or refresh-outcome target.

In [26]:
importance = pd.DataFrame({
    "feature": features,
    "importance": time_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
0,gsc_impressions,0.759139
3,gsc_avg_position,0.227536
2,ctr,0.012069
1,gsc_clicks,0.001256
4,ga4_pageviews,0.000000
5,ga4_sessions,0.000000
6,ga4_engaged_sessions,0.000000
7,sessions_organic,0.000000
8,sessions_direct,0.000000
9,sessions_referral,0.000000


In [28]:
# Section 4 — Current monitoring baseline

monitoring_snapshot = pd.DataFrame({
    "metric": [
        "Time-aware NDCG@10",
        "Rule baseline NDCG@10",
        "Zero next-day-click proportion",
        "Queue size",
        "Top feature",
        "Top feature importance",
    ],
    "value": [
        time_ndcg,
        baseline_ndcg,
        zero_click_rate,
        len(action_queue),
        importance.iloc[0]["feature"],
        importance.iloc[0]["importance"],
    ]
})

display(monitoring_snapshot)

print("\nCurrent reason-code distribution:")
print(
    action_queue["reason_codes"]
    .value_counts()
)

,metric,value
0,Time-aware NDCG@10,0.689987
1,Rule baseline NDCG@10,0.066051
2,Zero next-day-click proportion,0.95
3,Queue size,180
4,Top feature,gsc_impressions
5,Top feature importance,0.759139



Current reason-code distribution:
reason_codes
LOW_CTR                  105
LOW_CTR|WEAK_POSITION     47
MODEL_PRIORITY            16
TREND_DOWN|LOW_CTR        12
Name: count, dtype: int64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the paper

The final ranked queue and monitoring summary are exported to `work/outputs/`.

The queue contains pseudonymized identifiers, model ranking scores, observable performance signals, reason codes, and the recommended human-review action.

The export is intended to support the research paper and reproducibility. It should not be interpreted as a list of pages that are proven to require refreshing.

In [29]:
# Section 5 — Export artifacts for the paper

from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

queue_path = (
    output_dir /
    "ml10_ranked_action_queue.csv"
)

monitoring_path = (
    output_dir /
    "ml10_monitoring_snapshot.csv"
)

action_queue.to_csv(
    queue_path,
    index=False
)

monitoring_snapshot.to_csv(
    monitoring_path,
    index=False
)

print("Wrote:")
print("-", queue_path)
print("-", monitoring_path)

assert queue_path.exists()
assert monitoring_path.exists()

print(
    "\nExport verification: PASSED"
)

Wrote:
- work\outputs\ml10_ranked_action_queue.csv
- work\outputs\ml10_monitoring_snapshot.csv

Export verification: PASSED


In [30]:
# ML-10 final validation checks

assert len(action_queue) > 0
assert action_queue["priority_rank"].is_monotonic_increasing
assert action_queue["model_score"].notna().all()
assert action_queue["reason_codes"].notna().all()

assert queue_path.exists()
assert monitoring_path.exists()

print("=" * 60)
print("ML-10 CONTENT ACTION PLAYBOOK — FINAL CHECK")
print("=" * 60)

print("Queue rows:", len(action_queue))
print("Top-20 rows:", min(20, len(action_queue)))

print(
    "Model NDCG@10:",
    round(model_ndcg, 6)
)

print(
    "Rule baseline NDCG@10:",
    round(baseline_ndcg, 6)
)

print(
    "Zero-target proportion:",
    round(zero_click_rate, 4)
)

print(
    "\nExports:"
)

print("-", queue_path)
print("-", monitoring_path)

print(
    "\nAll ML-10 checks passed."
)

ML-10 CONTENT ACTION PLAYBOOK — FINAL CHECK
Queue rows: 180
Top-20 rows: 20
Model NDCG@10: 0.689987
Rule baseline NDCG@10: 0.066051
Zero-target proportion: 0.95

Exports:
- work\outputs\ml10_ranked_action_queue.csv
- work\outputs\ml10_monitoring_snapshot.csv

All ML-10 checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.